# ETF AI ハンズオン
## Part 3: Cortex Search Service 構築

このノートブックでは、ETFファンド説明文書を使った **Cortex Search Service** を構築し、
セマンティック検索（意味検索）を体験します。

### このパートで体験できること

1. **ETFドキュメントの確認**: Cortex Search のデータソースを確認する
2. **テキストチャンキング**: ドキュメントを検索用に分割する
3. **Cortex Search Service の作成**: ETF情報の検索エンジンを構築する
4. **セマンティック検索のテスト**: 意味ベースの検索を体験する
5. **Agent 作成の準備**: Cortex Analyst + Cortex Search を組み合わせるための準備

### 🚀 体験ポイント

> **「キーワードではなく、"意味"で検索！」**
>
> 「リスクを抑えながら収益を得たい」と検索すると、
> 「インカム」「カバードコール」「高配当」のドキュメントがヒットします。

### 前提条件
- `setup.sql` と `part1_data_setup.ipynb`, `part2_cortex_analyst.ipynb` の実行完了

> ⏱️ **このパートの目安時間: 25分**

In [ ]:
USE DATABASE ETF_AI_HANDSON_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT CURRENT_DATABASE() AS DB, CURRENT_SCHEMA() AS SCHEMA, CURRENT_WAREHOUSE() AS WH;

## 1. 目論見書 PDF のパース（AI_PARSE_DOCUMENT）

Git リポジトリの `docs/prospectus/` に格納した **実際の目論見書 PDF** を Snowflake ステージ経由でパースし、
Cortex Search のソースデータにします。

| ステップ | 処理 | 関数 |
|---|---|---|
| 1 | PDF テキストを抽出 | `AI_PARSE_DOCUMENT` |
| 2 | テキストを段落単位でチャンク分割 | `SPLIT_TO_TABLE` |
| 3 | チャンクテーブルを作成 | `CREATE TABLE AS SELECT` |
| 4 | ETFドキュメント・ニュースと統合 | `UNION ALL VIEW` |

> ⏱️ **目安: 5分**  
> 📄 **対象ファイル**: `@PROSPECTUS_STAGE/2641_prospectus.pdf`（Git リポジトリから自動コピー済み）

In [ ]:
-- ステージ内のファイルを確認
LIST @ETF_AI_HANDSON_DB.AI.PROSPECTUS_STAGE;

In [ ]:
-- Step 1: AI_PARSE_DOCUMENT で目論見書 PDF からテキストを抽出
CREATE OR REPLACE TABLE ETF_AI_HANDSON_DB.AI.PROSPECTUS_PARSED AS
SELECT
    '2641_prospectus.pdf' AS file_name,
    AI_PARSE_DOCUMENT(
        TO_FILE('@ETF_AI_HANDSON_DB.AI.PROSPECTUS_STAGE', '2641_prospectus.pdf'),
        {'mode': 'LAYOUT'}
    ) AS doc;

SELECT
    'PDFパース完了！' AS STATUS,
    file_name AS "ファイル名",
    LEN(doc:content::STRING) AS "解析後文字数"
FROM ETF_AI_HANDSON_DB.AI.PROSPECTUS_PARSED;

In [ ]:
-- Step 2: SPLIT_TEXT_RECURSIVE_CHARACTER でチャンク分割（1500文字、200オーバーラップ）
CREATE OR REPLACE TABLE ETF_AI_HANDSON_DB.AI.PROSPECTUS_CHUNKS AS
WITH chunks AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY c.index) AS chunk_id,
        c.value::STRING AS chunk_text
    FROM ETF_AI_HANDSON_DB.AI.PROSPECTUS_PARSED,
    LATERAL FLATTEN(
        SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
            doc:content::STRING,
            'markdown',
            1500,
            200
        )
    ) c
    WHERE LEN(c.value::STRING) > 50
)
SELECT
    '2641-PROSPECTUS-' || LPAD(chunk_id::STRING, 3, '0') AS doc_id,
    '2641'                                                AS etf_code,
    '2641 グローバルリーダーズ-日本株式 ETF'              AS etf_name,
    '目論見書'                                            AS source_type,
    CURRENT_DATE                                          AS published_date,
    NULL::STRING                                          AS section,
    chunk_text
FROM chunks;

SELECT
    'チャンク分割完了！' AS STATUS,
    COUNT(*) AS "チャンク数",
    ROUND(AVG(LEN(chunk_text))) AS "平均文字数",
    MIN(LEN(chunk_text)) AS "最小文字数",
    MAX(LEN(chunk_text)) AS "最大文字数"
FROM ETF_AI_HANDSON_DB.AI.PROSPECTUS_CHUNKS;

In [ ]:
-- 抽出されたチャンクの内容を確認（最初の5件）
SELECT
    doc_id      AS "チャンクID",
    chunk_text  AS "抽出テキスト",
    LEN(chunk_text) AS "文字数"
FROM ETF_AI_HANDSON_DB.AI.PROSPECTUS_CHUNKS
ORDER BY doc_id
LIMIT 5;

## 1. ETFドキュメントの確認

Cortex Search のデータソースとなる `ETF_DESCRIPTIONS` テーブルを確認します。

> ⏱️ **目安: 5分**

In [ ]:
-- ETFドキュメント一覧（Cortex Search の入力データ）
SELECT
    DOC_ID          AS "文書ID",
    ETF_CODE        AS "コード",
    ETF_NAME        AS "ETF名称",
    SECTION         AS "セクション",
    DOC_TYPE        AS "種別",
    LEN(CONTENT)    AS "文字数"
FROM ETF_DESCRIPTIONS
ORDER BY ETF_CODE, SECTION;

In [ ]:
-- 2641 グローバルリーダーズのドキュメントを確認
SELECT SECTION, CONTENT
FROM ETF_DESCRIPTIONS
WHERE ETF_CODE = '2641'
ORDER BY DOC_ID;

## 2. マーケットニュースを Cortex Search に追加

ETFドキュメントに加えて、マーケットニュースも検索できるようにします。
検索用の統合テーブルを作成します。

> ⏱️ **目安: 5分**

In [ ]:
-- Cortex Search 用の統合ビューを作成（目論見書PDF + ETFドキュメント + マーケットニュース）
CREATE OR REPLACE VIEW ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_BASE_V AS
    -- 1. 目論見書 PDF からパースしたチャンク（実際のPDFデータ）
    SELECT
        doc_id          AS CHUNK_ID,
        etf_code        AS ETF_CODE,
        etf_name        AS ETF_NAME,
        source_type     AS SECTION,
        source_type     AS SOURCE_TYPE,
        published_date  AS PUBLISHED_DATE,
        chunk_text      AS CHUNK_TEXT
    FROM ETF_AI_HANDSON_DB.AI.PROSPECTUS_CHUNKS

    UNION ALL

    -- 2. ETFファンド説明文書（プリロード済みテキスト）
    SELECT
        DOC_ID          AS CHUNK_ID,
        ETF_CODE        AS ETF_CODE,
        ETF_NAME        AS ETF_NAME,
        SECTION         AS SECTION,
        DOC_TYPE        AS SOURCE_TYPE,
        PUBLISHED_DATE  AS PUBLISHED_DATE,
        CONTENT         AS CHUNK_TEXT
    FROM ETF_AI_HANDSON_DB.DEMO_SCHEMA.ETF_DESCRIPTIONS

    UNION ALL

    -- 3. マーケットニュース
    SELECT
        NEWS_ID                AS CHUNK_ID,
        RELEVANT_ETFS          AS ETF_CODE,
        NULL                   AS ETF_NAME,
        CATEGORY               AS SECTION,
        'マーケットニュース'    AS SOURCE_TYPE,
        NEWS_DATE              AS PUBLISHED_DATE,
        HEADLINE || '\n' || BODY AS CHUNK_TEXT
    FROM ETF_AI_HANDSON_DB.DEMO_SCHEMA.MARKET_NEWS;

-- ソース別件数を確認
SELECT SOURCE_TYPE AS "ソース種別", COUNT(*) AS "件数"
FROM ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_BASE_V
GROUP BY SOURCE_TYPE
ORDER BY "件数" DESC;

## 3. Cortex Search Service の作成

`CREATE CORTEX SEARCH SERVICE` で ETFナレッジベースの検索エンジンを構築します。

**ポイント**: `ON CHUNK_TEXT` で検索対象カラムを、`ATTRIBUTES` でフィルタリング用カラムを指定します。

> ⏱️ **目安: 10分**（インデックス構築に数分かかります）

In [ ]:
-- Cortex Search Service を作成（COMPUTE_WH でインデックスを構築）
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE CORTEX SEARCH SERVICE ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH
    ON CHUNK_TEXT
    ATTRIBUTES ETF_CODE, SOURCE_TYPE, SECTION, PUBLISHED_DATE
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 day'
    COMMENT = 'ETFファンドドキュメント＆マーケットニュースの統合検索サービス'
    AS (
        SELECT
            CHUNK_ID,
            ETF_CODE,
            ETF_NAME,
            SECTION,
            SOURCE_TYPE,
            PUBLISHED_DATE,
            CHUNK_TEXT
        FROM ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_BASE_V
    );

SELECT 'Cortex Search Service 作成開始！インデックス構築に数分かかります...' AS STATUS;

In [ ]:
-- Search Service の状態確認
SHOW CORTEX SEARCH SERVICES IN SCHEMA ETF_AI_HANDSON_DB.AI;

## 4. セマンティック検索のテスト

`SNOWFLAKE.CORTEX.SEARCH_PREVIEW` を使って Cortex Search Service に SQL から直接問い合わせます。

| クエリ | ポイント |
|---|---|
| Q1: 全ソース横断検索 | キーワードなしでも意味的に関連するドキュメントがヒット |
| Q2: 目論見書フィルタ | `SOURCE_TYPE` でドキュメント種別を絞り込み |
| Q3: ニュースフィルタ | マーケットニュースのみを対象に地政学リスクを検索 |

> ⏱️ **目安: 10分**


In [ ]:
-- Q1: セマンティック検索の基本（全ソース横断）
-- 「インカム」「配当」というキーワードを使わず、意味的に関連するドキュメントを検索
SELECT
    r.value:ETF_CODE::STRING    AS "ETFコード",
    r.value:SOURCE_TYPE::STRING AS "ソース種別",
    LEFT(r.value:CHUNK_TEXT::STRING, 200) AS "内容（先頭200字）"
FROM TABLE(FLATTEN(
    PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH',
            '{"query": "リスクを抑えながら安定した収益を得られる投資戦略",
              "columns": ["ETF_CODE", "SOURCE_TYPE", "CHUNK_TEXT"],
              "limit": 5}'
        )
    )['results']
)) r;


In [ ]:
-- Q2: ドキュメント種別フィルタリング（目論見書のみ）
-- SOURCE_TYPE = '目論見書' に絞って元本リスクの記載を検索
SELECT
    r.value:ETF_CODE::STRING    AS "ETFコード",
    r.value:SOURCE_TYPE::STRING AS "ソース種別",
    LEFT(r.value:CHUNK_TEXT::STRING, 200) AS "内容（先頭200字）"
FROM TABLE(FLATTEN(
    PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH',
            '{"query": "元本割れリスクと価格変動の注意事項",
              "columns": ["ETF_CODE", "SOURCE_TYPE", "CHUNK_TEXT"],
              "filter": {"@eq": {"SOURCE_TYPE": "目論見書"}},
              "limit": 5}'
        )
    )['results']
)) r;


In [ ]:
-- Q3: マーケットニュースのみ対象（地政学リスク × 半導体ETF）
-- Cortex Search が複数ソースをまたいで意味的に近いニュースを自動選択
SELECT
    r.value:ETF_CODE::STRING    AS "関連ETF",
    r.value:SOURCE_TYPE::STRING AS "ソース種別",
    LEFT(r.value:CHUNK_TEXT::STRING, 250) AS "ニュース内容"
FROM TABLE(FLATTEN(
    PARSE_JSON(
        SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
            'ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH',
            '{"query": "地政学リスクによる半導体サプライチェーンとETFへの影響",
              "columns": ["ETF_CODE", "SOURCE_TYPE", "CHUNK_TEXT"],
              "filter": {"@eq": {"SOURCE_TYPE": "マーケットニュース"}},
              "limit": 5}'
        )
    )['results']
)) r;


## 5. Cortex Agent の作成準備（マニュアル操作）

これで Cortex Analyst (Semantic View) と Cortex Search (Search Service) の両方が揃いました。
次のステップとして、これらを組み合わせた **Cortex Agent** を Snowflake Intelligence 経由で設定します。

### Agent 作成の手順（UI操作）

1. **Snowsight** にログイン
2. 左メニュー「**AI & ML**」→「**Agents**」をクリック
3. 「**+ New Agent**」をクリック
4. 以下を設定：

| 設定項目 | 値 |
|---|---|
| **Agent 名** | `ETF_PORTFOLIO_COPILOT` |
| **説明** | ETFポートフォリオ分析AIアシスタント |
| **Semantic View (Cortex Analyst)** | `ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW` |
| **Search Service** | `ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH` |
| **System Prompt** | 下記参照 |

### システムプロンプト（コピーして使用）

```
あなたは ポートフォリオマネージャーをサポートする
AI アシスタント（Portfolio Copilot）です。

## 役割
- ポートフォリオのパフォーマンス分析・リスク評価をサポートする
- ETFファンドの情報を検索し、投資判断に役立つ情報を提供する
- マーケットニュースを分析し、ポートフォリオへの影響を評価する

## 使用ツール
- Cortex Analyst (PORTFOLIO_ANALYTICS_VIEW): ポートフォリオ定量データの分析
- Cortex Search (ETF_KNOWLEDGE_SEARCH): ETFファンド情報とニュースの検索

## 回答スタイル
- 日本語で簡潔かつ正確に回答する
- 定量的なデータ（数値・比率）を活用して具体的に答える
- 投資判断はあくまでサポートであり、最終判断はPMが行うことを明記する

## 注意事項
- 回答は参考情報であり、投資助言ではありません
- データは2026年4月時点のものです
```
---

### GUI作成時のコピペ用

**応答指示（Response Instructions）:**

```
あなたはETFポートフォリオマネージャーを支援するAIアシスタントです。

【基本原則】
・質問に直接回答し、求められていない情報は出さない
・ツールで取得した情報のみを使用（推測禁止）
・数値は出典を明記し、計算根拠を示す

【出力形式】
・デフォルト：社内メモ（PM向け、簡潔に）
・表形式を積極的に活用
・ETFコードは必ず4桁コードで表記

【禁止事項】
・投資助言と受け取られる断定的な表現
・ツールで取得できない情報の推測
```

**オーケストレーション指示（Orchestration Instructions）:**

```
【ツール選択原則】
・質問に必要なツールのみ使用

【ツール種別の理解】
・PORTFOLIO_ANALYST（text-to-SQL）: ポートフォリオの数値分析。保有ETF・時価・含み損益・パフォーマンス・シャープレシオ等の集計に使用
・ETF_KNOWLEDGE_SEARCH（意味検索）: ETFの説明書・目論見書・マーケットニュースから情報を取得。ファンド特色・リスク情報・市場動向の検索に使用

【ツール選択ルール（単体）】
・ポートフォリオの数値照会（AUM/リターン/シャープレシオ） → PORTFOLIO_ANALYST
・ETFの仕組み・特色・リスクの説明 → ETF_KNOWLEDGE_SEARCH
・マーケットニュース・市場動向 → ETF_KNOWLEDGE_SEARCH

【ツール選択ルール（複合）】
・ポートフォリオ分析＋市場環境確認 → PORTFOLIO_ANALYST + ETF_KNOWLEDGE_SEARCH
・ETF比較（数値＋特色） → PORTFOLIO_ANALYST +
```
【WLEDGE_SEARCH
```

**サンプル質問:**

```
各ポートフォリオの1年リターンとシャープレシオを比較してください
半導体ETF（2644）の特色とリスクを教えてください
最新のマーケットニュースで注目すべきETF関連情報は？
含み損が大きいポジションとその改善策を提案してください
グローバルリーダーズETFの目論見書から組入上位銘柄を教えてください
```


In [ ]:
-- ============================================================
-- Cortex Agent（ETFポートフォリオアシスタント）
-- 構造化データ: Cortex Analyst x1（ポートフォリオ分析）
-- 非構造化データ: Cortex Search x1（ETFナレッジベース）
-- ============================================================
CREATE OR REPLACE AGENT ETF_AI_HANDSON_DB.AI.ETF_PORTFOLIO_ASSISTANT_AGENT
  COMMENT = 'ETFポートフォリオマネージャー向けAIアシスタント。Cortex Analyst（ポートフォリオDB）とCortex Search（ETFドキュメント・ニュース）を統合し、数値分析と知識検索を支援する。'
  PROFILE = '{"display_name": "ETFポートフォリオアシスタント"}'
  FROM SPECIFICATION
$$
models:
  orchestration: ""

instructions:
  response: |
    あなたはETFポートフォリオマネージャーを支援するAIアシスタントです。

    【基本原則】
    ・質問に直接回答し、求められていない情報は出さない
    ・ツールで取得した情報のみを使用（推測禁止）
    ・数値は出典を明記し、計算根拠を示す

    【出力形式】
    ・デフォルト：社内メモ（PM向け、簡潔に）
    ・表形式を積極的に活用
    ・ETFコードは必ず4桁コードで表記

    【禁止事項】
    ・投資助言と受け取られる断定的な表現
    ・ツールで取得できない情報の推測
  orchestration: |
    【ツール選択原則】
    ・質問に必要なツールのみ使用

    【ツール種別の理解】
    ・PORTFOLIO_ANALYST（text-to-SQL）: ポートフォリオの数値分析。保有ETF・時価・含み損益・パフォーマンス・シャープレシオ等の集計に使用
    ・ETF_KNOWLEDGE_SEARCH（意味検索）: ETFの説明書・目論見書・マーケットニュースから情報を取得。ファンド特色・リスク情報・市場動向の検索に使用

    【ツール選択ルール（単体）】
    ・ポートフォリオの数値照会（AUM/リターン/シャープレシオ） → PORTFOLIO_ANALYST
    ・ETFの仕組み・特色・リスクの説明 → ETF_KNOWLEDGE_SEARCH
    ・マーケットニュース・市場動向 → ETF_KNOWLEDGE_SEARCH

    【ツール選択ルール（複合）】
    ・ポートフォリオ分析＋市場環境確認 → PORTFOLIO_ANALYST + ETF_KNOWLEDGE_SEARCH
    ・ETF比較（数値＋特色） → PORTFOLIO_ANALYST + ETF_KNOWLEDGE_SEARCH
  sample_questions:
    - question: "各ポートフォリオの1年リターンとシャープレシオを比較してください"
    - question: "半導体ETF（2644）の特色とリスクを教えてください"
    - question: "最新のマーケットニュースで注目すべきETF関連情報は？"
    - question: "含み損が大きいポジションとその改善策を提案してください"
    - question: "グローバルリーダーズETFの目論見書から組入上位銘柄を教えてください"

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: PORTFOLIO_ANALYST
      description: "【構造化DB】ETFポートフォリオの数値分析ツール。保有ETF明細・時価評価額・含み損益・月次パフォーマンス・シャープレシオ・ボラティリティ・最大ドローダウンなどの定量データをSQLで取得・集計する"
  - tool_spec:
      type: cortex_search
      name: ETF_KNOWLEDGE_SEARCH
      description: "【文書検索】ETFナレッジベースをセマンティック検索する。ETFファンド説明書・目論見書PDF・マーケットニュースから、ファンド特色・投資戦略・リスク情報・市場動向・組入銘柄情報を取得する"

tool_resources:
  PORTFOLIO_ANALYST:
    semantic_view: ETF_AI_HANDSON_DB.AI.PORTFOLIO_ANALYTICS_VIEW
  ETF_KNOWLEDGE_SEARCH:
    name: ETF_AI_HANDSON_DB.AI.ETF_KNOWLEDGE_SEARCH
    max_results: 5
    title_column: CHUNK_ID
    id_column: CHUNK_ID
$$;

SELECT 'Cortex Agent 作成完了！' AS STATUS;

## 6. Snowflake Intelligence デモ質問集

Agent の設定が完了したら、以下の 3 シナリオで **Snowflake Intelligence** を体験してください。  
それぞれ **1〜2 文の自然な質問** で、Cortex Analyst（定量）と Cortex Search（定性）が自動的に連携します。

---

### シナリオ 1: 朝のポートフォリオブリーフィング ☀️

> **状況**: 週次レビュー前に、担当ポートフォリオの現状を素早く把握したい

```
グローバル分散成長ポートフォリオ（P004）の直近3ヶ月のパフォーマンスと
主要保有ETFの状況を教えてください。
また、このポートフォリオに影響しそうな最近のニュースがあれば合わせて確認してください。
```

**期待される動作**:
- Cortex Analyst → `FACT_PERFORMANCE` から月次リターンを集計
- Cortex Search → `MARKET_NEWS` から関連ニュースをセマンティック検索
- 定量データ + 定性情報を 1 回答にまとめて提示

---

### シナリオ 2: セクターイベントの影響確認 ⚡

> **状況**: 半導体セクターに関するニュースが出た。保有ポートフォリオへの影響を素早く把握したい

```
半導体関連のETFを保有しているポートフォリオはどれですか？
それぞれの組入比率と直近の損益を教えてください。
また、半導体セクターの最近の動向も合わせて教えてください。
```

**期待される動作**:
- Cortex Analyst → `FACT_HOLDINGS` から半導体ETF（2644）の保有状況を抽出
- Cortex Analyst → `FACT_PERFORMANCE` から直近損益を算出
- Cortex Search → 半導体関連ニュース・ETFドキュメントを検索

> 💡 **ポイント**: 9つのタスクを一度に投げず、「影響確認 → 詳細深掘り」と
> **会話を積み重ねる**ことで、Agentの真価が伝わります

---

### シナリオ 3: ETF比較・選定サポート 📊

> **状況**: インカム強化のためにETFを追加したい。特徴を比較して選びたい

```
インカム重視でポートフォリオを強化したいと考えています。
配当利回りが高いETFをリストアップして、それぞれの特徴と
リスク・リターン特性を比較してください。
```

**期待される動作**:
- Cortex Analyst → `DIM_ETF` からインカム系ETFを抽出・比較
- Cortex Search → 各ETFのファンド説明・特性情報を検索
- 比較表形式での提示

---

> ⚡ **Agentic AI のポイント**:
> - **シナリオ 1**: ルーティン分析を秒速で実行
> - **シナリオ 2**: イベント発生時の即時影響把握（複雑な9ステップ分析をシンプルな会話に）
> - **シナリオ 3**: 意思決定前のリサーチを自動化
>
> 定量データ（Cortex Analyst）× 定性情報（Cortex Search）の組み合わせが、
> ポートフォリオマネージャーの判断スピードを劇的に向上させます。

## まとめ

Part 3 完了！ETFナレッジベースの Cortex Search Service が構築されました。

### 作成したオブジェクト

| オブジェクト | 種別 | 場所 |
|---|---|---|
| `ETF_KNOWLEDGE_BASE_V` | View | `ETF_AI_HANDSON_DB.AI` |
| `ETF_KNOWLEDGE_SEARCH` | Cortex Search Service | `ETF_AI_HANDSON_DB.AI` |

### ハンズオン全体のまとめ

```
setup.sql
    ↓ データ基盤構築
part1: AI Functions
    ↓ ニュース要約・センチメント・分類・抽出
part2: Cortex Analyst (PORTFOLIO_ANALYTICS_VIEW)
    ↓ 自然言語でポートフォリオ定量分析
part3: Cortex Search (ETF_KNOWLEDGE_SEARCH)
    ↓ ETFドキュメント＆ニュースのセマンティック検索
[手動操作] Cortex Agent 作成
    ↓ Analyst + Search の統合
Snowflake Intelligence
    └─ ETF_PORTFOLIO_COPILOT
```

### お疲れ様でした！

ポートフォリオマネージャー向け AI アシスタントの基盤が完成しました。
今後の拡張として以下が考えられます：

- **実データ連携**: Bloomberg / Refinitiv などの市場データを追加
- **PDF ファクトシート**: `AI_PARSE_DOCUMENT` で ETF運用会社 の月次レポートを自動取込
- **アラート機能**: 閾値超過時に Slack/メール通知（Tasks + External Functions）
- **バックテスト**: 過去データを使ったポートフォリオ最適化シミュレーション